In [75]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
GOOGLE_API_KEY = ""
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
)
llm.invoke(input="say hi")

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e227e-b44f-77c1-b336-e3e9a82c1ab1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 10, 'total_tokens': 13, 'input_token_details': {'cache_read': 0}})

In [77]:
# ── 2. Database connection ────────────────────────────────────────
from langchain_community.utilities import SQLDatabase
db_user     = "dev"
db_password = "dev123"
db_host     = "localhost"
db_name     = "atliq_tshirts"
port = 3306

db = SQLDatabase.from_uri(
    f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",
    sample_rows_in_table_info=3,
)
print(db.table_info)


CREATE TABLE discounts (
	discount_id INTEGER NOT NULL AUTO_INCREMENT, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount DECIMAL(5, 2), 
	PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), 
	CONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))
)COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB DEFAULT CHARSET=utf8mb4

/*
3 rows from discounts table:
discount_id	t_shirt_id	pct_discount

*/


CREATE TABLE t_shirts (
	t_shirt_id INTEGER NOT NULL AUTO_INCREMENT, 
	brand ENUM('Van Huesen','Levi','Nike','Adidas') NOT NULL, 
	color ENUM('Red','Blue','Black','White') NOT NULL, 
	size ENUM('XS','S','M','L','XL') NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_chk_1 CHECK ((`price` between 10 and 50))
)COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB DEFAULT CHARSET=utf8mb4

/*
3 rows from t_shirts table:
t_shirt_id	brand	color	size	price	stock_quantity
1	Adidas	Red	M	40	3

In [79]:
# ── 3. Helper: run a natural-language question against the DB ─────
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_classic.chains import create_sql_query_chain

In [80]:
# Simple chain (no few-shot) – used to collect answers for few_shots dict
_basic_chain = create_sql_query_chain(llm, db)
_execute     = QuerySQLDataBaseTool(db=db, verbose=True)

In [81]:
def run_question(question: str) -> str:
    sql   = _basic_chain.invoke({"question": question})
    return _execute.invoke(sql)

run_question('show me all  nike t_shirts   left?')

[(15, 'Nike', 'Red', 'XS', 32, 41), (56, 'Nike', 'Red', 'S', 30, 60), (31, 'Nike', 'Red', 'L', 18, 12), (41, 'Nike', 'Blue', 'M', 31, 22), (17, 'Nike', 'Blue', 'L', 47, 42), (32, 'Nike', 'Blue', 'XL', 38, 14), (23, 'Nike', 'Black', 'XS', 18, 59), (11, 'Nike', 'Black', 'S', 45, 23), (58, 'Nike', 'Black', 'XL', 49, 97), (4, 'Nike', 'White', 'XS', 22, 96), (13, 'Nike', 'White', 'S', 17, 39), (66, 'Nike', 'White', 'M', 11, 56), (3, 'Nike', 'White', 'XL', 27, 64)]

"[(15, 'Nike', 'Red', 'XS', 32, 41), (56, 'Nike', 'Red', 'S', 30, 60), (31, 'Nike', 'Red', 'L', 18, 12), (41, 'Nike', 'Blue', 'M', 31, 22), (17, 'Nike', 'Blue', 'L', 47, 42), (32, 'Nike', 'Blue', 'XL', 38, 14), (23, 'Nike', 'Black', 'XS', 18, 59), (11, 'Nike', 'Black', 'S', 45, 23), (58, 'Nike', 'Black', 'XL', 49, 97), (4, 'Nike', 'White', 'XS', 22, 96), (13, 'Nike', 'White', 'S', 17, 39), (66, 'Nike', 'White', 'M', 11, 56), (3, 'Nike', 'White', 'XL', 27, 64)]"

In [84]:
import os
from langchain_openai import OpenAI
api_key = os.getenv("OPENAI_KEY")
llm = OpenAI(api_key = api_key, temperature=0.2)

In [85]:
# ── 3. Helper: run a natural-language question against the DB ─────
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_classic.chains import create_sql_query_chain
import re

def clean_sql(sql: str) -> str:
    """Strip any LLM preamble like 'SQLQuery:' from the output."""
    sql = re.sub(r'(?i)^sqlquery:\s*', '', sql.strip())
    sql = re.sub(r'```sql|```', '', sql).strip()
    return sql

_basic_chain = create_sql_query_chain(llm, db)
_execute     = QuerySQLDataBaseTool(db=db)

def run_question(question: str) -> str:
    sql = _basic_chain.invoke({"question": question})
    #sql = clean_sql(sql)
    print(f"SQL: {sql}")
    return _execute.invoke(sql)

def run_sql(sql: str) -> str:
    return _execute.invoke(sql)

# ── 4. Test ───────────────────────────────────────────────────────
print(run_question("how many brand of t-shirts do we have?"))

SQL: SELECT COUNT(DISTINCT brand) AS "Number of Brands" FROM t_shirts
[(4,)]


In [87]:
# ── 4. Collect answers that will populate few-shot examples ───────
qns1 = run_question(
    "How many t-shirts do we have left for Nike in XS size and white color?"
)

# The LLM may mis-calculate inventory price, so we run the explicit query:
qns2 = run_sql(
    "SELECT SUM(price*stock_quantity) FROM t_shirts WHERE size = 'S'"
)

# Revenue post-discount for Levi's – explicit query avoids hallucinated columns:
qns3 = run_sql("""
    SELECT SUM(a.total_amount * ((100 - COALESCE(discounts.pct_discount, 0)) / 100))
           AS total_revenue
    FROM (
        SELECT SUM(price * stock_quantity) AS total_amount, t_shirt_id
        FROM t_shirts
        WHERE brand = 'Levi'
        GROUP BY t_shirt_id
    ) a
    LEFT JOIN discounts ON a.t_shirt_id = discounts.t_shirt_id
""")

qns4 = run_sql(
    "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'"
)

qns5 = run_sql(
    "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand = 'Levi' AND color = 'White'"
)

SQL: SELECT COUNT(*) FROM t_shirts WHERE brand = 'Nike' AND size = 'XS' AND color = 'White'


In [88]:
print(run_question('How much is the total price of the inventory for all S-size t-shirts?'))

SQL: SELECT SUM(price) AS total_price FROM t_shirts WHERE size = 'S'
[(Decimal('345'),)]


In [89]:
few_shots = [
    {
        "Question":  "How many t-shirts do we have left for Nike in XS size and white color?",
        "SQLQuery":  "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'",
        "SQLResult": "Result of the SQL query",
        "Answer":    qns1,
    },
    {
        "Question":  "How much is the total price of the inventory for all S-size t-shirts?",
        "SQLQuery":  "SELECT SUM(price*stock_quantity) FROM t_shirts WHERE size = 'S'",
        "SQLResult": "Result of the SQL query",
        "Answer":    qns2,
    },
    {
        "Question":  "If we have to sell all the Levi's T-shirts today with discounts applied. How much revenue our store will generate (post discounts)?",
        "SQLQuery":  """SELECT SUM(a.total_amount * ((100 - COALESCE(discounts.pct_discount, 0)) / 100)) AS total_revenue
FROM (
    SELECT SUM(price * stock_quantity) AS total_amount, t_shirt_id
    FROM t_shirts WHERE brand = 'Levi' GROUP BY t_shirt_id
) a LEFT JOIN discounts ON a.t_shirt_id = discounts.t_shirt_id""",
        "SQLResult": "Result of the SQL query",
        "Answer":    qns3,
    },
    {
        "Question":  "If we have to sell all the Levi's T-shirts today. How much revenue our store will generate without discount?",
        "SQLQuery":  "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'",
        "SQLResult": "Result of the SQL query",
        "Answer":    qns4,
    },
    {
        "Question":  "How many white color Levi's shirt I have?",
        "SQLQuery":  "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand = 'Levi' AND color = 'White'",
        "SQLResult": "Result of the SQL query",
        "Answer":    qns5,
    },
]

In [109]:

# ── 6. Semantic-similarity example selector ───────────────────────
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

In [110]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
to_vectorize = [" ".join(example.values()) for example in few_shots]
vectorstore = Chroma.from_texts(to_vectorize, embeddings, metadatas=few_shots)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5659.14it/s]


In [ ]:
example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vectorstore,
    k=2,
)

# Quick sanity-check
print(example_selector.select_examples({"Question": "How many Adidas T shirts I have left in my store?"}))

[{'Question': 'How many t-shirts do we have left for Nike in XS size and white color?', 'SQLQuery': "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'", 'Answer': '[(1,)]', 'SQLResult': 'Result of the SQL query'}, {'Question': 'How many t-shirts do we have left for Nike in XS size and white color?', 'SQLResult': 'Result of the SQL query', 'SQLQuery': "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'", 'Answer': '[(1,)]'}]


In [104]:
mysql_prompt = """You are a MySQL expert. Given an input question, first create a syntactically correct MySQL query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per MySQL. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in backticks (`) to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use CURDATE() function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: Query to run with no pre-amble
SQLResult: Result of the SQLQuery
Answer: Final answer here

No pre-amble.
"""

In [115]:
# ── 7. Prompt templates ───────────────────────────────────────────
from langchain_community.agent_toolkits.sql.prompt import SQL_PREFIX, SQL_SUFFIX

PROMPT_SUFFIX = """Only use the following tables:
{table_info}

Question: {input}"""

example_prompt = PromptTemplate(
    input_variables=["Question", "SQLQuery", "SQLResult", "Answer"],
    template="\nQuestion: {Question}\nSQLQuery: {SQLQuery}\nSQLResult: {SQLResult}\nAnswer: {Answer}",
)

few_shot_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix=mysql_prompt,
    suffix=PROMPT_SUFFIX,
    input_variables=["input", "table_info", "top_k"],
)

In [116]:
# Updated
from langchain_classic.chains import create_sql_query_chain  # ← fix this
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# create_sql_query_chain accepts a custom prompt
few_shot_chain = create_sql_query_chain(llm, db, prompt=few_shot_prompt)

def ask(question: str) -> str:
    """Generate SQL with few-shot prompt, execute it, return the answer."""
    sql    = few_shot_chain.invoke({"question": question})
    result = _execute.invoke(sql)
    print(f"\nQuestion : {question}\nSQL      : {sql}\nResult   : {result}\n")
    return result

# ── 9. Test queries ───────────────────────────────────────────────
ask("How many white color Levi's shirt I have?")
ask("How much is the price of the inventory for all small size t-shirts?")
ask("How much is the price of all white color Levi t-shirts?")
ask("If we have to sell all the Nike's T-shirts today with discounts applied. How much revenue our store will generate (post discounts)?")
ask("If we have to sell all the Van Heusen T-shirts today with discounts applied. How much revenue our store will generate (post discounts)?")
ask("How much revenue our store will generate by selling all Van Heusen T-Shirts without discount?")


Question : How many white color Levi's shirt I have?
SQL      : SELECT SUM(stock_quantity) FROM t_shirts WHERE brand = 'Levi' AND color = 'White'
Result   : [(Decimal('294'),)]


Question : How much is the price of the inventory for all small size t-shirts?
SQL      : SELECT SUM(price * stock_quantity) AS total_inventory_price
FROM t_shirts WHERE size = 'S'
Result   : [(Decimal('17661'),)]


Question : How much is the price of all white color Levi t-shirts?
SQL      : SELECT SUM(price * stock_quantity) AS total_price
FROM t_shirts WHERE brand = 'Levi' AND color = 'White'
Result   : [(Decimal('9915'),)]


Question : If we have to sell all the Nike's T-shirts today with discounts applied. How much revenue our store will generate (post discounts)?
SQL      : SELECT SUM(a.total_amount * ((100 - COALESCE(discounts.pct_discount, 0)) / 100)) AS total_revenue
FROM (
    SELECT SUM(price * stock_quantity) AS total_amount, t_shirt_id
    FROM t_shirts WHERE brand = 'Nike' GROUP BY t_shirt_id
) 

"[(Decimal('28053'),)]"

In [ ]:
import subprocess
result = subprocess.run(['grep', '-r', 'create_sql_query_chain', 
                       '/home/ubrillo/genai-play/venv-genai/lib/'], 
                       capture_output=True, text=True)
print(result.stdout[:2000])

/home/ubrillo/genai-play/venv-genai/lib/python3.12/site-packages/langchain_classic/chains/sql_database/query.py:def create_sql_query_chain(
/home/ubrillo/genai-play/venv-genai/lib/python3.12/site-packages/langchain_classic/chains/sql_database/query.py:        from langchain_classic.chains import create_sql_query_chain
/home/ubrillo/genai-play/venv-genai/lib/python3.12/site-packages/langchain_classic/chains/sql_database/query.py:        chain = create_sql_query_chain(model, db)
/home/ubrillo/genai-play/venv-genai/lib/python3.12/site-packages/langchain_classic/chains/__init__.py:    "create_sql_query_chain": "langchain_classic.chains.sql_database.query",

